# 第 6 讲练习：链表

建议画节点和箭头。修改 `.next` 前，先确认后续节点仍然可以找到。

In [1]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def build_linked_list(values):
    dummy = ListNode()
    tail = dummy
    for value in values:
        tail.next = ListNode(value)
        tail = tail.next
    return dummy.next

def linked_list_values(head, limit=100):
    result = []
    while head is not None and len(result) < limit:
        result.append(head.val)
        head = head.next
    return result

In [2]:
values = [1, 2, 3, 4, 5]
linked_list = build_linked_list(values)
print(linked_list_values(linked_list))

[1, 2, 3, 4, 5]


In [6]:
linked_list.next

## 1. 概念题

1. 为什么链表不能 `O(1)` 访问第 `i` 个节点？
2. 反转链表时为什么必须先保存 `current.next`？
3. 虚拟头节点解决了什么特殊情况？
4. 为什么判断环时比较 `slow is fast`，而不是比较节点值？
5. 单链表已知前驱节点时删除其后继的复杂度是多少？

在这里作答。

答：  
1.因为链表的查询依赖前边的节点，只能够按照顺序查询。  
2.因为要把previous赋值给current.next，必须先保存一下原来的下个节点   
3.保证了第一个节点不会被删除   
4不同的节点可以保存相同的val   
5.O（1）

## 2. 删除指定值

删除链表中所有值等于 `target` 的节点，返回新头节点。要求 `O(n)` 时间、`O(1)` 额外空间。

提示：使用虚拟头节点，并思考删除后前驱指针是否应该前进。

In [10]:
def remove_elements(head, target):
    dummy = ListNode(0,head)
    current = dummy
    while current.next is not None:
        if current.next.val == target:
            current.next = current.next.next
        else:
            current =  current.next
    return dummy.next

In [11]:
assert linked_list_values(remove_elements(build_linked_list([]), 1)) == []
assert linked_list_values(remove_elements(build_linked_list([1,1,2,3,1]), 1)) == [2,3]
assert linked_list_values(remove_elements(build_linked_list([1,1]), 1)) == []
assert linked_list_values(remove_elements(build_linked_list([1,2,3]), 9)) == [1,2,3]

## 3. 反转链表

原地反转单链表并返回新头。编码前说明 `previous`、`current`、`next_node` 分别指向哪一部分。

current 当前正在处理的node
previous 原本的链表中的前一个node
next 接下来要处理的node


In [12]:
def reverse_list(head):
    current = head
    previous = None
    while current is not None:
        next = current.next
        current.next = previous
        previous = current
        current = next
    return previous

In [13]:
assert linked_list_values(reverse_list(build_linked_list([]))) == []
assert linked_list_values(reverse_list(build_linked_list([1]))) == [1]
assert linked_list_values(reverse_list(build_linked_list([1,2,3,4]))) == [4,3,2,1]

## 4. 合并两个有序链表

复用原节点，合并两个非递减链表并返回结果头节点。要求 `O(m+n)` 时间、`O(1)` 额外空间。

dummy.next 到 tail 是已经确定的有序前缀；list1 和 list2 分别指向两个链表尚未合并的第一个节点。

In [14]:
def merge_two_lists(list1, list2):
    dummy = ListNode()
    tail = dummy

    while list1 is not None and list2 is not None:
        if list1.val < list2.val:
            tail.next = list1
            list1 = list1.next
        else:
            tail.next = list2
            list2 = list2.next
        
        tail = tail.next
    if list1 is None:
        tail.next = list2
    else:
        tail.next = list1

    return dummy.next


In [15]:
assert linked_list_values(merge_two_lists(build_linked_list([]), build_linked_list([]))) == []
assert linked_list_values(merge_two_lists(build_linked_list([]), build_linked_list([1,2]))) == [1,2]
assert linked_list_values(merge_two_lists(build_linked_list([1,2,4]), build_linked_list([1,3,4]))) == [1,1,2,3,4,4]

## 5. 链表中点

返回中间节点；如果有两个中间节点，返回第二个。要求只遍历一趟，不计算链表长度。

使用快慢指针，fast一次走两步，slow一次走一步   
fast走到末尾时，slow

In [18]:
def middle_node(head):
    slow = head
    fast = head
    while fast is not None and fast.next is not None:
        slow = slow.next
        fast = fast.next.next
    return slow

In [19]:
assert middle_node(build_linked_list([1])).val == 1
assert middle_node(build_linked_list([1,2,3,4,5])).val == 3
assert middle_node(build_linked_list([1,2,3,4,5,6])).val == 4

## 6. 挑战：环形链表

判断链表中是否存在环，要求 `O(1)` 额外空间。不能使用 `set`。

slow 每轮走 1 步；   
fast 每轮走 2 步；  
如果有环必然会相遇   



In [22]:
def has_cycle(head):
    slow = head
    fast = head

    while fast is not None and fast.next is not None:
        slow = slow.next
        fast = fast.next.next
        if slow == fast:
            return True
    
    return False

In [23]:
assert has_cycle(None) is False
assert has_cycle(build_linked_list([1,2,3])) is False

a = ListNode(1)
b = ListNode(2)
c = ListNode(3)
a.next = b
b.next = c
c.next = b
assert has_cycle(a) is True

single = ListNode(9)
single.next = single
assert has_cycle(single) is True

# 累计复习

## 7. 哈希复习：附近重复元素

如果存在两个相同元素，其下标距离不超过 `k`，返回 `True`。要求 `O(n)` 时间。

提示：`dict` 可以保存某个值最近一次出现的下标。

In [31]:
def contains_nearby_duplicate(nums, k):
    dict = {}
    minlength = 1000000
    for i,x in enumerate(nums):
        
        if x in dict:
            length = i-dict[x]
            minlength = min(length,minlength)
        dict[x] = i # 储存下标
    
    if minlength <= k:
        return True
    else:
        return False

In [32]:
assert contains_nearby_duplicate([1,2,3,1], 3) is True
assert contains_nearby_duplicate([1,0,1,1], 1) is True
assert contains_nearby_duplicate([1,2,3,1,2,3], 2) is False
assert contains_nearby_duplicate([], 2) is False

## 8. 队列复习：逐层传播

给定邻接表和起点，返回恰好距离起点 `k` 条边的所有节点。输出顺序不作要求。使用 BFS。

In [46]:
from collections import deque
def nodes_at_distance(graph, start, k):
    que = deque([start])
    visit = {start: 0}

    while que:
        if visit[que[0]] == k:
            return list(que)
        current = que.popleft()

        for x in graph[current]:
            if x not in visit:
                visit[x] = visit[current] + 1
                que.append(x)

    return []




In [47]:
graph = {0:[1,2], 1:[0,3], 2:[0,4], 3:[1], 4:[2]}
assert set(nodes_at_distance(graph, 0, 0)) == {0}
assert set(nodes_at_distance(graph, 0, 1)) == {1,2}
assert set(nodes_at_distance(graph, 0, 2)) == {3,4}
assert nodes_at_distance(graph, 0, 5) == []

## 9. 总结

1. 哪些题使用了 dummy？
2. 反转链表时三个引用分别表示什么？
3. 快慢指针循环条件为什么要先检查 `fast`？
4. 哪些操作比较节点身份而不是值？
5. 累计复习题是否还能独立识别哈希和 BFS？

答：   
1.删除node和合并两个链表的题   
2.current是这个循环中要处理的node，next是下一个循环中要处理的node，previous是current的前一个节点   
3.因为只有fast本身存在，那么fast.next才有意义   
4.判断环、是否相交这种，必须要求两个节点是同一个节点  
5.哈希的题可以做出来，queue的题有些忘记了